# Training and Evaluation
This notebook trains a logistic regression model on the cleaned churn dataset, tunes the regularization parameter with grid search, finds an optimal classification threshold, and reports evaluation metrics.


In [59]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_curve, roc_auc_score)

df = pd.read_csv("../data/clean_data.csv")
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=50
)


In [60]:
param_grid = {"C": [0.01, 0.1, 1, 10, 15, 20, 25, 30, 40, 50, 100]}

grid = GridSearchCV(
    LogisticRegression(class_weight="balanced", solver="liblinear", random_state=42),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("Best parameters:", grid.best_params_)


Best parameters: {'C': 0.1}


In [61]:
y_prob = best_model.predict_proba(X_test)[:, 1]
best_threshold = 0.0
best_f1 = 0.0

for threshold in np.arange(0.1, 0.91, 0.01):
    y_pred = (y_prob >= threshold).astype(int)
    current_f1 = f1_score(y_test, y_pred)
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best F1 score:", best_f1)


Best threshold: 0.5799999999999997
Best F1 score: 0.6200676437429538


In [62]:
y_pred = (y_prob >= best_threshold).astype(int)
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("Classification report:", classification_report(y_test, y_pred))
print("Confusion matrix:", confusion_matrix(y_test, y_pred))


Accuracy: 0.7608232789212207
ROC AUC: 0.8303624480095068
Classification report:               precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.74      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.76      0.77      1409

Confusion matrix: [[797 238]
 [ 99 275]]


## Optional model experiments
The cells below are placeholders for alternative classifier experiments. Uncomment and adapt these models as needed for comparison.


In [ ]:
# from sklearn.tree import DecisionTreeClassifier
# dt = DecisionTreeClassifier(random_state=42, class_weight="balanced")
# dt.fit(X_train, y_train)
# y_pred_dt = dt.predict(X_test)
# print("Decision tree F1:", f1_score(y_test, y_pred_dt))
# print("Classification report:\n", classification_report(y_test, y_pred_dt))

Decision tree F1: 0.4665757162346521
Classification report:
               precision    recall  f1-score   support

           0       0.81      0.82      0.81      1035
           1       0.48      0.46      0.47       374

    accuracy                           0.72      1409
   macro avg       0.64      0.64      0.64      1409
weighted avg       0.72      0.72      0.72      1409



In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
# rf.fit(X_train, y_train)
# y_pred_rf = rf.predict(X_test)
# print("Random forest F1:", f1_score(y_test, y_pred_rf))
# print("Classification report:\n", classification_report(y_test, y_pred_dt))

Random forest F1: 0.589242053789731
Classification report:
               precision    recall  f1-score   support

           0       0.81      0.82      0.81      1035
           1       0.48      0.46      0.47       374

    accuracy                           0.72      1409
   macro avg       0.64      0.64      0.64      1409
weighted avg       0.72      0.72      0.72      1409



In [70]:
import pickle

with open("../model/churn_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [71]:
with open("../model/threshold.pkl", "wb") as f:
    pickle.dump(best_threshold, f)

In [72]:
with open("../model/features.pkl", "wb") as f:
    pickle.dump(list(X.columns), f)

In [3]:
import pickle

with open("../model/features.pkl", "rb") as f:
    features = pickle.load(f)

print(type(features))
print(len(features))
print(features)

<class 'list'>
31
['senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'paperless_billing', 'monthly_charges', 'total_charges', 'contract_Month-to-month', 'contract_One year', 'contract_Two year', 'gender_Male', 'payment_method_Credit card (automatic)', 'payment_method_Electronic check', 'payment_method_Mailed check', 'multiple_lines_No phone service', 'multiple_lines_Yes', 'internet_service_Fiber optic', 'internet_service_No', 'online_security_No internet service', 'online_security_Yes', 'online_backup_No internet service', 'online_backup_Yes', 'device_protection_No internet service', 'device_protection_Yes', 'tech_support_No internet service', 'tech_support_Yes', 'streaming_t_v_No internet service', 'streaming_t_v_Yes', 'streaming_movies_No internet service', 'streaming_movies_Yes']
